In [ ]:
import rasterio
import geopandas as gpd
import numpy as np
from rasterio.mask import mask

# ---------------- 사용자 입력 ----------------
tif_path = r"E:\SRTMSDM-DSM.tif"
shp_path = r"D:\projects\03_DTM\Bolivia\shp\AOI2.shp"

# ① Shapefile 로드 (좌표계 일치 확인)
gdf = gpd.read_file(shp_path)

# ② GeoTIFF 열기
with rasterio.open(tif_path) as src:
    # 좌표계 맞추기 (필요 시)
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)

    # ③ 마스크 적용하여 폴리곤 내부 값 추출
    geoms = gdf.geometry.values  # geometry list
    out_image, out_transform = mask(dataset=src, shapes=geoms, crop=True)
    out_data = out_image[0]  # 단일 밴드일 경우

    # ④ NoData 값 제외한 유효값만 선택
    valid_mask = (out_data != src.nodata) & (~np.isnan(out_data))
    values = out_data[valid_mask]

    # ⑤ 통계 계산
    if values.size > 0:
        min_val = np.min(values)
        max_val = np.max(values)
        mean_val = np.mean(values)
        print(f"📊 MIN: {min_val:.4f}, MAX: {max_val:.4f}, MEAN: {mean_val:.4f}")
    else:
        print("⚠️ 선택한 영역 내 유효한 픽셀이 없습니다.")


In [ ]:
import rasterio
import geopandas as gpd
import numpy as np
from rasterio.mask import mask

# ---------------- 사용자 입력 ----------------
tif_path = r"E:\SRTMSDM-DSM.tif"
shp_path = r"D:\projects\03_DTM\Bolivia\shp\AOI2.shp"
THRESHOLD = -1000
P_MIN = 2
P_MAX = 98

# ---------------- AOI 불러오기 ----------------
gdf = gpd.read_file(shp_path)

# ---------------- TIF 열기 및 AOI 마스킹 ----------------
with rasterio.open(tif_path) as src:
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)

    geoms = gdf.geometry.values
    out_image, out_transform = mask(dataset=src, shapes=geoms, crop=True)
    out_data = out_image[0]
    nodata = src.nodata

    # 유효한 값만 필터링 (-1000 초과 + NoData 제외)
    if nodata is not None:
        valid_mask = (out_data != nodata) & (out_data > THRESHOLD)
    else:
        valid_mask = out_data > THRESHOLD

    valid_values = out_data[valid_mask]

# ---------------- Percentile 클리핑 ----------------
if valid_values.size > 0:
    lower = np.percentile(valid_values, P_MIN)
    upper = np.percentile(valid_values, P_MAX)

    clipped_values = valid_values[(valid_values >= lower) & (valid_values <= upper)]

    # ---------------- 통계 계산 ----------------
    if clipped_values.size > 0:
        min_val = np.min(clipped_values)
        max_val = np.max(clipped_values)
        mean_val = np.mean(clipped_values)

        print(f"📊 Percentile {P_MIN}~{P_MAX}% 기준 통계:")
        print(f"   ▶ CLIPPED MIN:  {min_val:.4f}")
        print(f"   ▶ CLIPPED MAX:  {max_val:.4f}")
        print(f"   ▶ CLIPPED MEAN: {mean_val:.4f}")
        print(f"   ▶ 원래 픽셀 수: {valid_values.size}, 클리핑 후: {clipped_values.size}")
    else:
        print("⚠️ 클리핑된 범위에 유효 픽셀이 없습니다.")
else:
    print("⚠️ AOI 내 -1000 초과 유효 픽셀이 없습니다.")
